In [ ]:
# -*- coding: utf-8 -*-
"""monte_carlo_ES_simple_politica_optima_01.ipynb

Automatically generated by Colab.

"""

# =====================================================================
# Algoritmo de Control Montecarlo con Primera Visita
# (sin matplotlib)
# =====================================================================
# _Aprendizaje por Refuerzo I
# _Maestria en Inteligencia Artificial
# _UBA
# _2025
# =====================================================================

"""

Robot limpiador de un parque
 _________
|_|_|_|_|_|
|_|_|_|_|_|
|_|_|_|_|_|
|_|_|_|_|_|
|_|_|_|_|_|<--Contenedor

"""

In [1]:
import numpy as np
import random
import collections
from tqdm import tqdm

# --- Configuración del Entorno REDUCIDO ---
GRID_ROWS = 5  # Reducido
GRID_COLS = 5  # Reducido
CONTAINER_POS = (4, 4)  # celda objetivo (contenedor)

# obstaculos (lista de tuplas (fila, columna)) - ajjustados al grid 5x5
OBSTACLES = [
    (1, 1),
    (1, 2),
    (1, 3),  # fila superior de obstaculos
    (2, 1),  # folumna izquierda de obstaculos
    (3, 1),
    (3, 3),  # obstaculos inferiores
]
OBSTACLE_SET = set(OBSTACLES)  # mas eficiente para búsquedas

ACTIONS = {0: "UP", 1: "DOWN", 2: "LEFT", 3: "RIGHT"}
ACTION_VECTORS = {  # cambios en (dr, dc) para cada accion
    0: (-1, 0),  # UP
    1: (1, 0),  # DOWN
    2: (0, -1),  # LEFT
    3: (0, 1),  # RIGHT
}
N_ACTIONS = len(ACTIONS)

# --- parametros MC ES ---
GAMMA = 1.0
# episodios
NUM_EPISODES = 500000
MAX_STEPS_PER_EPISODE = GRID_ROWS * GRID_COLS * 2  # límite ajustado


# --- funciones Auxiliares ---
def is_valid(state):
    """Comprueba si el estado está dentro de los límites del grid."""
    r, c = state
    return 0 <= r < GRID_ROWS and 0 <= c < GRID_COLS


def take_step(state, action):
    """
    Simula un paso en el entorno.
    Devuelve (next_state, reward, is_terminal).
    """
    if state == CONTAINER_POS:
        return state, 0, True  # ya está en el terminal

    dr, dc = ACTION_VECTORS[action]
    next_r, next_c = state[0] + dr, state[1] + dc
    next_state = (next_r, next_c)

    # comprobar limites y obstaculos
    if not is_valid(next_state) or next_state in OBSTACLE_SET:
        next_state = state  # chocó, se queda en el mismo lugar
        reward = -1  # penalizacion por intentar moverse (y chocar)
        is_terminal = False
    elif next_state == CONTAINER_POS:
        reward = 2  # recompensa (o ausencia de penalización) por llegar
        is_terminal = True
    else:
        reward = -0.1  # Costo de dar un paso valido
        is_terminal = False

    return next_state, reward, is_terminal


# --- Algoritmo Monte Carlo ES (Control) ---

# 1. Inicializacion
Q = collections.defaultdict(lambda: np.zeros(N_ACTIONS))
policy = collections.defaultdict(
    lambda: random.randint(0, N_ACTIONS - 1)
)  # Política inicial aleatoria
returns_sum = collections.defaultdict(float)
returns_count = collections.defaultdict(float)

#  estados validos para los Inicios Exploratorios
valid_start_states = []
for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        state = (r, c)
        if state != CONTAINER_POS and state not in OBSTACLE_SET:
            valid_start_states.append(state)
#  hay estados válidos para iniciar?
if not valid_start_states:
    raise ValueError("No valid start states found. Check obstacle/container placement.")


print(f"Ejecutando Control Monte Carlo ES (Grid 5x5) por {NUM_EPISODES} episodios...")

# 2. Bucle por cada episodio
for episode_num in tqdm(range(NUM_EPISODES)):
    # --- Simulación del Episodio con INICIO EXPLORATORIO ---
    #  estado s0 y acción a0 iniciales al azar
    start_state = random.choice(valid_start_states)
    start_action = random.randrange(N_ACTIONS)

    # ejecucion de la primera acción forzada
    current_state, reward, terminated = take_step(start_state, start_action)
    #  *primer* par (s,a) y la recompensa resultante R1
    episode_history = [(start_state, start_action, reward)]
    steps = 1

    # Generacion del resto de la trayectoria siguiendo la política actual pi
    while not terminated and steps < MAX_STEPS_PER_EPISODE:
        current_action = policy[current_state]
        next_state, reward, terminated = take_step(current_state, current_action)
        episode_history.append((current_state, current_action, reward))
        current_state = next_state
        steps += 1
    # if steps == MAX_STEPS_PER_EPISODE: print("Warning: Max steps reached")

    # --- Actualizacion de Q y Politica (GPI) ---
    G = 0
    pairs_visited_in_episode = set()

    for t in range(len(episode_history) - 1, -1, -1):
        state_t, action_t, reward_tp1 = episode_history[t]
        G = GAMMA * G + reward_tp1
        pair = (state_t, action_t)
        if pair not in pairs_visited_in_episode:
            returns_sum[pair] += G
            returns_count[pair] += 1
            Q[state_t][action_t] = returns_sum[pair] / returns_count[pair]
            policy[state_t] = np.argmax(Q[state_t])
            pairs_visited_in_episode.add(pair)

Ejecutando Control Monte Carlo ES (Grid 5x5) por 500000 episodios...


100%|██████████| 500000/500000 [00:13<00:00, 36048.42it/s]


In [2]:
# --- muestra de Resultados en una matrix ---
print("\n--- Entrenamiento Completado ---")
print("\nPolítica Óptima Aprendida (π) para Grid 5x5:")

action_symbols = {0: "^", 1: "v", 2: "<", 3: ">"}  # Simbolos para las acciones
grid_display = [[" " for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]

for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        state = (r, c)
        if state == CONTAINER_POS:
            grid_display[r][c] = "C"  # contenedor
        elif state in OBSTACLE_SET:
            grid_display[r][c] = "#"  # obstaculo
        elif state in policy:
            best_action_idx = policy[state]
            grid_display[r][c] = action_symbols[best_action_idx]
        else:
            # si un estado valido nunca fue inicio o visitado, podría no tener politica
            # muetra '.' o la accion aleatoria inicial si la tuviera
            default_action = collections.defaultdict(lambda: random.randint(0, N_ACTIONS - 1))[
                state
            ]
            grid_display[r][c] = action_symbols.get(default_action, ".")


# print la cuadrícula por consola
print("+" + "---+" * GRID_COLS)
for row in grid_display:
    print("| " + " | ".join(row) + " |")
    print("+" + "---+" * GRID_COLS)


--- Entrenamiento Completado ---

Política Óptima Aprendida (π) para Grid 5x5:
+---+---+---+---+---+
| < | ^ | > | > | v |
+---+---+---+---+---+
| > | # | # | # | v |
+---+---+---+---+---+
| < | # | v | > | v |
+---+---+---+---+---+
| < | # | v | # | v |
+---+---+---+---+---+
| > | > | > | > | C |
+---+---+---+---+---+
